In [7]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB


def dataset_path(name):
    for folder in (Path.cwd(), *Path.cwd().parents):
        path = folder / "practical_labs" / name
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find practical_labs/{name}")


# Part A: spam classification
emails = pd.read_csv(dataset_path("Lab2_dataset.csv"))
X_train, X_test, y_train, y_test = train_test_split(
    emails["text"].fillna(""), emails["label"], test_size=0.2,
    random_state=42, stratify=emails["label"]
)
vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=3000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

models = {"GaussianNB": GaussianNB(), "MultinomialNB": MultinomialNB()}
results = {}
for name, model in models.items():
    Xtr = X_train_vec.toarray() if name == "GaussianNB" else X_train_vec
    Xte = X_test_vec.toarray() if name == "GaussianNB" else X_test_vec
    model.fit(Xtr, y_train)
    predictions = model.predict(Xte)
    results[name] = (model, predictions, accuracy_score(y_test, predictions),
                     f1_score(y_test, predictions, pos_label="spam"))

print("Test-set performance (accuracy and spam F1)")
print(pd.DataFrame({name: {"accuracy": r[2], "spam F1": r[3]} for name, r in results.items()}).T.round(3))
best_name = max(results, key=lambda name: results[name][2])
print(f"Better model by accuracy: {best_name}")
print("\nBest-model classification report:")
print(classification_report(y_test, results[best_name][1], zero_division=0))
print("Confusion matrix (ham, spam):")
print(confusion_matrix(y_test, results[best_name][1], labels=["ham", "spam"]))
print("\nSample predictions:")
print(pd.DataFrame({
    "email": X_test.iloc[:8].str.replace(r"\\s+", " ", regex=True).str.slice(0, 110).to_numpy(),
    "actual": y_test.iloc[:8].to_numpy(),
    "predicted": results[best_name][1][:8],
}).to_string(index=False))


Test-set performance (accuracy and spam F1)
               accuracy  spam F1
GaussianNB        0.970    0.949
MultinomialNB     0.941    0.902
Better model by accuracy: GaussianNB

Best-model classification report:
              precision    recall  f1-score   support

         ham       0.98      0.97      0.98       735
        spam       0.94      0.96      0.95       300

    accuracy                           0.97      1035
   macro avg       0.96      0.97      0.96      1035
weighted avg       0.97      0.97      0.97      1035

Confusion matrix (ham, spam):
[[716  19]
 [ 12 288]]

Sample predictions:
                                                                                                           email actual predicted
 Subject: final cp name change and merger report 08 / 00\njust to clarify , i was in the february 2000 report in    ham       ham
                                     Subject: calpine daily gas nomination\n- hidalgo daily gas nomination . doc    ham     

In [8]:
# Part B: Airbnb nightly-price outliers
# outliers were identified using three methods: Mean ± 3 SD, Median ± 1.5 IQR, and 1st–99th percentiles
listings = pd.read_csv(dataset_path("AB_NYC_2019.csv"))
valid_prices = listings.loc[listings["price"] > 0].copy()
price = valid_prices["price"]
mean, std = price.mean(), price.std()
q1, median, q3 = price.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
lower_iqr, upper_iqr = q1 - 1.5 * iqr, q3 + 1.5 * iqr
lower_pct, upper_pct = price.quantile([0.01, 0.99])
methods = pd.DataFrame({
    "method": ["Mean ± 3 SD", "Median ± 1.5 IQR", "1st–99th percentiles"],
    "lower bound": [max(0, mean - 3 * std), lower_iqr, lower_pct],
    "upper bound": [mean + 3 * std, upper_iqr, upper_pct],
})
methods["outliers removed"] = [
    ((price < row["lower bound"]) | (price > row["upper bound"])).sum()
    for _, row in methods.iterrows()
]
print("\nPrice distribution before cleaning:")
print(price.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(2))
print("\nOutlier method comparison (non-positive prices excluded first):")
print(methods.round(2).to_string(index=False))

airbnb_clean = valid_prices.loc[price.between(lower_iqr, upper_iqr)].copy()
print(f"\nClean dataset using median/IQR fences: {len(airbnb_clean):,} of {len(listings):,} listings retained; "
      f"{len(listings) - len(airbnb_clean):,} removed.")
print("Cleaned price summary:")
print(airbnb_clean["price"].describe().round(2))
print("Remaining IQR outliers:", int((~airbnb_clean["price"].between(lower_iqr, upper_iqr)).sum()))


Price distribution before cleaning:
count    48884.00
mean       152.76
std        240.17
min         10.00
1%          30.00
25%         69.00
50%        106.00
75%        175.00
99%        799.00
max      10000.00
Name: price, dtype: float64

Outlier method comparison (non-positive prices excluded first):
              method  lower bound  upper bound  outliers removed
         Mean ± 3 SD          0.0       873.27               388
    Median ± 1.5 IQR        -90.0       334.00              2972
1st–99th percentiles         30.0       799.00               867

Clean dataset using median/IQR fences: 45,912 of 48,895 listings retained; 2,983 removed.
Cleaned price summary:
count    45912.00
mean       120.00
std         68.13
min         10.00
25%         65.00
50%        100.00
75%        159.00
max        334.00
Name: price, dtype: float64
Remaining IQR outliers: 0
